[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HumbertoDiego/AjustamentoBasicoIME/blob/main/10_deteccao_erros_grosseiros.ipynb)


# Ajustamento Básico - Detecção de erros grosseiros

**Maj Diego - 2° Semestre / 2026**

**Objetivos:**

1. Diferenciar erro aleatório, erro sistemático e erro grosseiro no contexto do ajustamento.
2. Formular testes global e local para controle de qualidade.
3. Aplicar resíduos padronizados e *data snooping* de Baarda.
4. Interpretar confiabilidade interna e externa de uma rede.

**Referências:**

- Ghilani, C. D. (2017). *Adjustment computations: Spatial data analysis* (6th ed.). Wiley. $\rightarrow$ **Cap. 5, 10 e 13**
- Teunissen, P. J. G. (2000). *Testing theory: An introduction*. Delft University Press.


## O Problema

O MMQ distribui erros aleatórios entre as observações, mas não sabe, sozinho, distinguir uma observação apenas ruidosa de uma observação contaminada por erro grosseiro.

Se uma observação errada permanece no ajustamento, ela pode deslocar os parâmetros, inflar a variância a posteriori e mascarar outros resíduos.


## 1. O que é erro grosseiro?

Um **erro grosseiro** é uma falha incompatível com o modelo estocástico adotado.

Exemplos:

- leitura anotada em estação errada;
- altura de prisma digitada incorretamente;
- ambiguidade GNSS mal fixada;
- visada feita para o alvo errado;
- observação afetada por multipercurso severo.


### **1.1 Ideia central da detecção**

Após o ajustamento, os resíduos devem ser compatíveis com as precisões declaradas.

A lógica é:

1. ajustar o modelo;
2. testar se o conjunto está estatisticamente coerente;
3. localizar observações suspeitas;
4. remover, corrigir ou reponderar com justificativa técnica.


## 2. Teste global do ajustamento

Para observações independentes e pesos coerentes, a estatéstica

$$
T = V^T P V
$$

segue aproximadamente uma distribuição qui-quadrado com $gl=m-n$ graus de liberdade, sob a hipótese nula:

$$
H_0: \sigma_0^2 = \hat{\sigma}_0^2
$$


### **2.1 Interpretação**

- Se $T$ está dentro da região de aceitação, o conjunto é globalmente compatível.
- Se $T$ é grande demais, há indício de erro grosseiro, modelo funcional inadequado ou modelo estocástico otimista.
- Se $T$ é pequeno demais, as precisões informadas podem estar pessimistas.

O teste global acusa problema, mas não informa qual observação é culpada.


**Exercício 01 (resolvido):** Um ajustamento tem $m=12$ observações, $n=5$ parâmetros e $V^TPV=18{,}6$. Teste a qualidade global com $\alpha=5\%$.


In [ ]:
from scipy.stats import chi2

m, n = 12, 5
gl = m - n
T = 18.6
alpha = 0.05

crit_inf = chi2.ppf(alpha/2, gl)
crit_sup = chi2.ppf(1-alpha/2, gl)

print(f"gl = {gl}")
print(f"Intervalo de aceitação: [{crit_inf:.3f}, {crit_sup:.3f}]")
print(f"T = {T:.3f}")
print("Rejeita H0?", T < crit_inf or T > crit_sup)


## 3. Resíduos padronizados

O resíduo bruto $v_i$ não deve ser comparado diretamente entre observações com precisões diferentes.

Usa-se o resíduo padronizado:

$$
w_i = \frac{v_i}{\sigma_{v_i}}
$$

em que $\sigma_{v_i}$ é o desvio padrão do resíduo da observação $i$.


### **3.1 Matriz cofatora dos resíduos**

No modelo paramétrico linear:

$$
Q_V = P^{-1} - A(A^TPA)^{-1}A^T
$$

Logo:

$$
\Sigma_V = \hat{\sigma}_0^2 Q_V
$$

A diagonal de $\Sigma_V$ fornece as variâncias dos resíduos.


## 4. Data snooping de Baarda

O *data snooping* aplica testes locais aos resíduos padronizados.

Uma regra prática comum é marcar como suspeita a observação com:

$$
|w_i| > c
$$

em que $c$ costuma ficar próximo de 3 para triagens introdutórias.


**Exercício 02 (resolvido):** Ajuste uma reta e identifique a observação mais suspeita pelos resíduos padronizados.


In [ ]:
import numpy as np

x = np.array([0, 1, 2, 3, 4, 5], dtype=float)
y = np.array([1.0, 2.1, 3.0, 8.2, 5.1, 6.0], dtype=float)
sigma = np.ones_like(y) * 0.2

A = np.column_stack([x, np.ones_like(x)])
L = y.reshape(-1, 1)
P = np.diag(1 / sigma**2)

Xa = np.linalg.inv(A.T @ P @ A) @ A.T @ P @ L
V = A @ Xa - L
Qv = np.linalg.inv(P) - A @ np.linalg.inv(A.T @ P @ A) @ A.T
sigv = np.sqrt(np.diag(Qv)).reshape(-1, 1)
w = V / sigv

print("Xa =", Xa.ravel())
print("Resíduos padronizados:")
for i, wi in enumerate(w.ravel(), start=1):
    print(f"obs {i}: w = {wi:.2f}")
print("Observação mais suspeita:", np.argmax(np.abs(w)) + 1)


## 5. Confiabilidade interna e externa

**Confiabilidade interna**: capacidade da rede de detectar erros grosseiros nas observações.

**Confiabilidade externa**: efeito de um erro não detectado sobre os parâmetros ajustados.

Uma rede pode ter boa precisão, mas baixa confiabilidade se uma observação crítica tiver pouca redundância.


### **5.1 Redundância local**

A redundância local mede quanto uma observação é controlada pelo restante da rede.

Observações com baixa redundância local tendem a esconder erros, pois seus resíduos ficam pequenos mesmo quando a observação está contaminada.


## 6. Fluxo recomendado de controle

1. Ajustar com o modelo funcional e estocástico escolhidos.
2. Aplicar teste global.
3. Se reprovado, verificar resíduos padronizados.
4. Investigar tecnicamente a observação mais suspeita.
5. Corrigir, remover ou reponderar.
6. Reajustar e documentar a decisão.


## Lista de exercícios complementares

**Exercício 03:** Simule uma rede de nivelamento com 6 desníveis e contamine uma observação com erro grosseiro. Aplique teste global e resíduos padronizados.

**Exercício 04:** Compare a detecção de um mesmo erro grosseiro em duas geometrias: uma rede com alta redundância e outra com baixa redundância.

**Exercício 05:** Em uma trilateração 2D, insira erro em uma distância e avalie o impacto sobre as coordenadas ajustadas.
